In [5]:
import pandas as pd 
import numpy as np


In [2]:
import pandas as pd
import os

# Use backslashes for Windows paths
data_path = r"/kaggle/input/datasets/pradyumansharmaaa/mercari/train.tsv"

# Check if file exists and is readable
if os.path.isfile(data_path):
    try:
        df = pd.read_csv(data_path, sep="\t")
        print(f"File loaded successfully!")
    except PermissionError:
        print(f"Permission denied. Check file permissions at: {data_path}")
else:
    print(f"File not found: {data_path}")
    print("Check if the path exists or adjust the directory name.")

File loaded successfully!


TypeError: read_csv() missing 1 required positional argument: 'filepath_or_buffer'

In [3]:
df.head()

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,NaN,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,NaN,35.0,1,New with tags. Leather horses. Retail for [rm]...
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,NaN,44.0,0,Complete with certificate of authenticity


In [ ]:
df.info()

In [ ]:
df['category_name'].value_counts()

In [ ]:
df['price'].describe()

In [ ]:
df[df['price'] > 500].count()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df['price'], bins=50)
plt.show()

In [ ]:
df[df['price']==0].count()

In [ ]:
## removing the datapoint with zero price

df_clean = df[df['price'] >0]

In [6]:
## applying log transform to data as it is right skewed

df['log_price'] = np.log1p(df['price'])



In [7]:
# First, look at WHAT these items are

df[df['price'] > 500][['name', 'category_name', 'price']].sort_values('price', ascending=False).head(5)

## expensive itemssss 

,name,category_name,price
760469,NEW Chanel WOC Caviar Gold Hardware,Women/Women's Handbags/Shoulder Bag,2009.0
1262245,NEW-Chanel Boy Wallet o Chain WOC Caviar,Women/Women's Handbags/Messenger & Crossbody,2006.0
1393600,David Yurman Wheaton ring,Women/Jewelry/Rings,2004.0
1250053,Brand New Chanel CC Quilted WOC,Women/Women's Handbags/Messenger & Crossbody,2000.0
742113,Chanel Chevron Fuschia Pink 2,Women/Women's Handbags/Shoulder Bag,2000.0


## Dealing With NaNs of Brand_Names 

In [8]:
print(df['brand_name'].isnull().sum())
print(df['brand_name'].isnull().mean()*100,"%")


632682
42.675687251902986 %


In [ ]:
df.isnull().sum()
df.isnull().mean()*100

In [9]:
### now let's create more features from the cateogory for our TF-idf

df['cat_main']=df['category_name'].str.split('/').str[0]
df['cat_sub']=df['category_name'].str.split('/').str[1]
df['cat_detail']=df['category_name'].str.split('/').str[2]


In [10]:
df['name_len']=df['name'].str.len()
df['desc_len']=df['item_description'].str.len()
df['name_len']=df['item_description'].str.split().str.len()

##3 longer description mean more valuablle item(a note for me also ;/)


In [11]:
## now increase the input singnals 

df['has_brand'] = df['brand_name'].notnull().astype(int)
df['brand_name'] = df['brand_name'].fillna("unknown")


In [12]:
## validation of above logic 

df.groupby('has_brand')['price'].median()

has_brand
0    14.0
1    20.0
Name: price, dtype: float64

In [13]:
df.groupby('cat_main')['price'].median().sort_values(ascending = False)

cat_main
Men                       21.0
Women                     19.0
Home                      18.0
Sports & Outdoors         16.0
Vintage & Collectibles    16.0
Beauty                    15.0
Electronics               15.0
Kids                      14.0
Other                     14.0
Handmade                  12.0
Name: price, dtype: float64

In [11]:
df.groupby('cat_sub')['price'].median().sort_values(ascending = False).tail(15)

cat_sub
Paper Ephemera               11.0
Cell Phones & Accessories    11.0
Books                        11.0
Accessories                  10.0
Artwork                      10.0
Art                          10.0
Books and Zines              10.0
Media                        10.0
Geekery                      10.0
Needlecraft                  10.0
Children                      9.0
Magazines                     9.0
Quilts                        8.0
Trading Cards                 8.0
Paper Goods                   6.0
Name: price, dtype: float64

In [12]:
df.groupby('cat_detail')['price'].median().sort_values(ascending = False).head(15)

cat_detail
Standard                       145.0
Air Conditioners               131.0
Lightweight                    105.0
Laptops & Netbooks             100.0
Women's Golf Clubs             100.0
Travel Systems                  95.0
Satchel                         90.0
Desktops & All-In-Ones          89.5
Oils & Fluids                   85.0
Lenses & Filters                70.0
Track & Sweat Suits             66.0
Cell Phones & Smartphones       65.0
Wind & Woodwind Instruments     64.0
Digital Cameras                 61.0
Feather Beds                    60.0
Name: price, dtype: float64

In [13]:

## checking cardinaltiy

print(df['cat_main'].nunique())  ### 10 OHE
print(df['cat_sub'].nunique())  ### Target Encoding replacing with mean
print(df['cat_detail'].nunique()) ### Target Encoding replacing mean

10
113
870


In [14]:
### what is this standard

df[df['cat_detail']=='Standard'][['name','price','cat_main']].head(10)

,name,price,cat_main
11730,Minnie Mouse Stroller,50.0,Kids
73876,Baby Jogger City Premiere,275.0,Kids
144452,Mamas and Papas Sola,150.0,Kids
380115,Bugaboo bee limited edition pendleton,76.0,Kids
383116,2015 uppababy vista,480.0,Kids
427316,Mama and papa Urbo 2 stroller,230.0,Kids
446858,Orbit Baby G3 Black Stroller Seat-NEW,145.0,Kids
452110,UppaBaby Jake Bassinet New Model,70.0,Kids
769140,G2 Orbit Baby Stroller Seat,65.0,Kids
894690,HOLD for LESLIE Nuna IVVI Stroller,315.0,Kids


In [15]:
## target map

cat_sub_map = df.groupby('cat_sub')['price'].mean()
cat_detail_map = df.groupby('cat_detail')['price'].mean()

df['cat_sub_encoded'] = df['cat_sub'].map(cat_sub_map)
df['cat_detail_encoded'] = df['cat_detail'].map(cat_detail_map)


In [16]:
df.drop(columns=['cat_sub_encoded', 'cat_detail_encoded'], inplace=True)

In [17]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['price', 'log_price'])
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.2, random_state=42
)


X_test

In [49]:
cat_sub_map = df.groupby('cat_sub')['price'].mean()

cat_detail_map = df.groupby('cat_detail')['price'].mean()

In [50]:

X_train['cat_sub_encoded']    = X_train['cat_sub'].map(cat_sub_map)
X_test['cat_sub_encoded']     = X_test['cat_sub'].map(cat_sub_map)

X_train['cat_detail_encoded'] = X_train['cat_detail'].map(cat_detail_map)
X_test['cat_detail_encoded']  = X_test['cat_detail'].map(cat_detail_map)

In [20]:

## fliing test data with global mean
global_mean = y_train.mean
X_test['cat_sub_encoded'] = X_test['cat_sub_encoded'].fillna(global_mean)
X_test['cat_detail_encoded'] = X_test['cat_detail_encoded'].fillna(global_mean)

In [30]:
# Check if cat_main columns exist
print([col for col in X_train.columns if 'cat_main' in col])

# Also check all columns
print(X_train.columns.tolist())

['cat_main']
['train_id', 'name', 'item_condition_id', 'category_name', 'brand_name', 'shipping', 'item_description', 'cat_main', 'cat_sub', 'cat_detail', 'name_len', 'desc_len', 'has_brand', 'cat_sub_encoded', 'cat_detail_encoded']


In [31]:
X_train

,train_id,name,item_condition_id,category_name,brand_name,shipping,item_description,cat_main,cat_sub,cat_detail,name_len,desc_len,has_brand,cat_sub_encoded,cat_detail_encoded
1416089,1416089,LuLaRoe kids L/XL leggings,3,Kids/Boys (4+)/Bottoms,unknown,1,Worn once. Still in great condition,Kids,Boys (4+),Bottoms,6.0,35.0,0,27.849325,16.833355
1423955,1423955,Bundle 5 Display mannequins,1,Other/Other/Other,unknown,0,Brand new,Other,Other,Other,2.0,9.0,0,24.865364,23.694827
403867,403867,LIVING PROOF PERFECT HAIR DAY DRY SHAMPO,1,Beauty/Hair Care/Styling Products,unknown,0,This listing is for 3 full size bottles of liv...,Beauty,Hair Care,Styling Products,86.0,463.0,0,19.374646,16.191356
701974,701974,Palazzo pants,2,Women/Pants/Casual Pants,unknown,0,Like new adorable black and white palazzo pant...,Women,Pants,Casual Pants,111.0,589.0,0,19.645373,18.400573
1124330,1124330,RESERVED FOR Ms Jas PINK BOYSHORTS LARGE,1,Women/Underwear/Panties,PINK,1,NEW WITH TAGS MORNING SKY SHEER SEAFOAM OLIVE ...,Women,Underwear,Panties,9.0,51.0,1,18.097813,17.200451
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,259178,Brooks brothers girls,3,Kids/Girls (4+)/Dresses,Brooks Brothers,0,"Beautiful dress, another of her favorites. Sz....",Kids,Girls (4+),Dresses,11.0,73.0,1,18.552418,18.824669
1414414,1414414,LulaRoe Randy size Large,1,Women/Tops & Blouses/Knit Top,unknown,0,"Brand new, never worn or washed, size Large, N...",Women,Tops & Blouses,Knit Top,13.0,83.0,0,18.237514,19.711157
131932,131932,This sale is,1,Kids/Toys/Dolls & Accessories,unknown,0,American girl doll Tenney. Comes with the doll...,Kids,Toys,Dolls & Accessories,18.0,97.0,0,21.522112,26.984882
671155,671155,Iphone headphone lightning cable split,1,Electronics/Cell Phones & Accessories/Cables &...,Apple,0,No description yet,Electronics,Cell Phones & Accessories,Cables & Adapters,3.0,18.0,1,30.142278,10.941572


## MOdel Training

In [21]:
X_train = pd.get_dummies(X_train,columns=['cat_main'],drop_first=True)
X_test = pd.get_dummies(X_test,columns=['cat_main'],drop_first=True)

# Align columns between train and test
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## Adding more features

In [30]:
X_train['name_len']        = X_train['name'].str.len()
X_train['desc_len']        = X_train['item_description'].str.len()
X_train['name_word_count'] = X_train['name'].str.split().str.len()
X_train['desc_word_count'] = X_train['item_description'].str.split().str.len()

X_test['name_len']         = X_test['name'].str.len()
X_test['desc_len']         = X_test['item_description'].str.len()
X_test['name_word_count']  = X_test['name'].str.split().str.len()
X_test['desc_word_count']  = X_test['item_description'].str.split().str.len()

## Aplllying TF-IDF

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack,csr_matrix
import numpy as np



In [31]:
### filling every Nan if any 

X_train['name'] = X_train['name'].fillna('')
X_test['name']  = X_test['name'].fillna('')

X_train['item_description'] = X_train['item_description'].fillna('')
X_test['item_description']  = X_test['item_description'].fillna('')

In [32]:
### TF-idf on name

tfidf_name = TfidfVectorizer(max_features=5000, ngram_range=(1,1))
X_train_name = tfidf_name.fit_transform(X_train['name'])
X_test_name  = tfidf_name.transform(X_test['name'])

In [33]:
# Step 3: TF-IDF on description
tfidf_desc = TfidfVectorizer(max_features=5000, ngram_range=(1,1))
X_train_desc = tfidf_desc.fit_transform(X_train['item_description'])
X_test_desc  = tfidf_desc.transform(X_test['item_description'])

In [54]:
import numpy as np
from scipy.sparse import hstack, csr_matrix

# Convert to float32 explicitly
X_train_num = csr_matrix(X_train[features].astype(np.float32).values)
X_test_num  = csr_matrix(X_test[features].astype(np.float32).values)

# Then stack
X_train_final = hstack([X_train_num, X_train_name, X_train_desc])
X_test_final  = hstack([X_test_num,  X_test_name,  X_test_desc])

In [53]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

In [55]:
features = [
    'item_condition_id',
    'shipping',
    'has_brand',
    'cat_sub_encoded',
    'cat_detail_encoded'
]
features += ['name_len','desc_len','name_word_count',"desc_word_count"]
cat_main_cols = [col for col in X_train.columns if col.startswith('cat_main_')]
features += cat_main_cols

In [56]:
model = lgb.LGBMRegressor(
    n_estimators=500,
    random_state=42,
    device='gpu',
    force_row_wise=True,
    verbose=100  # prints progress every 100 trees so you know it's running
)
model.fit(X_train_final, y_train)

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.999556


Exception ignored on calling ctypes callback function: <function _log_callback at 0x7ccae1982520>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


Total Bins 1681806
[LightGBM] [Info] Number of data points in the train set: 1186028, number of used features: 88986
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 10 dense feature groups (13.57 MB) transferred to GPU in 0.014848 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 2.978479
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 11
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 11
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] [Debug]

...

Exception ignored on calling ctypes callback function: <function _log_callback at 0x7ccae1982520>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


Trained a tree with leaves = 31 and depth = 29


Exception ignored on calling ctypes callback function: <function _log_callback at 0x7ccae1982520>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


Trained a tree with leaves = 31 and depth = 25
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 28
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 26
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 28
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 30
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 29
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 29
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 28
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 29
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 26
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 29
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 27
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 29
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 24
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 27
[LightGBM] [Debug] Trained a 

LGBMRegressor(device='gpu', force_row_wise=True, n_estimators=500,
              random_state=42, verbose=100)

In [58]:
y_pred = model.predict(X_test_final)

# Competition metric (RMSLE) - log scale RMSE
rmsle = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSLE (competition metric): {rmsle:.4f}")

# Human readable - actual dollar error
actual_rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))
print(f"RMSE (actual $): {actual_rmse:.2f}")

RMSLE (competition metric): 0.4895
RMSE (actual $): 29.98


TypeError: float() argument must be a string or a real number, not 'method'

In [43]:
for col in features:
    # Try converting all at once and see
    try:
        test_convert = X_train[features].astype(np.float32)
        print("Conversion OK ✅")
        print(test_convert.dtypes)
    except Exception as e:
        print(f"Error: {e}")
    

Conversion OK ✅
item_condition_id                  float32
shipping                           float32
has_brand                          float32
cat_sub_encoded                    float32
cat_detail_encoded                 float32
name_len                           float32
desc_len                           float32
name_word_count                    float32
desc_word_count                    float32
cat_main_Electronics               float32
cat_main_Handmade                  float32
cat_main_Home                      float32
cat_main_Kids                      float32
cat_main_Men                       float32
cat_main_Other                     float32
cat_main_Sports & Outdoors         float32
cat_main_Vintage & Collectibles    float32
cat_main_Women                     float32
dtype: object
Conversion OK ✅
item_condition_id                  float32
shipping                           float32
has_brand                          float32
cat_sub_encoded                    float32
cat_deta

In [45]:
print(type(X_train_name))
print(type(X_train_desc))
print(X_train_name.dtype)
print(X_train_desc.dtype)

<class 'scipy.sparse._csr.csr_matrix'>
<class 'scipy.sparse._csr.csr_matrix'>
float64
float64


Step 1 OK ✅ (1186028, 18)
Step 2 OK ✅ float32
Step 3 OK ✅ (1186028, 18)
Step 4 OK ✅ (296507, 18)
Step 5 OK ✅ (1186028, 100018)
Step 6 OK ✅ (296507, 100018)


PROBLEM COLUMN: cat_sub_encoded → float() argument must be a string or a real number, not 'method'
PROBLEM COLUMN: cat_detail_encoded → float() argument must be a string or a real number, not 'method'
